# Patient Simulator Debug Notebook

Interactive notebook for chatting with different patient simulators and inspecting their system prompts.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import json
import yaml
from pathlib import Path

sys.path.append("..")

from patient_simulator.misc.llm import OpenRouterLLM, VLLM
from patient_simulator.misc.utils import parse_transcript_file
from patient_simulator.patients import (
    PatientsWithPersonality,
    BaselinePatient,
    CraftMDPatient,
)
from patient_simulator.misc.utils import create_llm_instance

## Initialize LLMs

In [3]:
openrouter_key = json.loads(Path("../keys.json").read_text())["OPENROUTER_KEY"]

In [4]:
# !pkill -f VLLM::EngineCore

In [5]:
meta_llm = VLLM(
    model="mistralai/Ministral-3-14B-Instruct-2512",
    engine_kwargs={
        "tensor_parallel_size": 1,
        "gpu_memory_utilization": 0.8,
        "max_model_len": 1024,
    },
    sampling_kwargs={
        "temperature": 0.05,
        # "top_p": 1.0,
        "max_tokens": 1024,
    },
)

Using cached model from: /home/mschlager/models/hf/mistralai__Ministral-3-14B-Instruct-2512
Loading LLM model: /home/mschlager/models/hf/mistralai__Ministral-3-14B-Instruct-2512...


[2026-04-11 10:39:52] INFO tekken.py:195: Non special vocabulary size is 130072 with 1000 special tokens.
[2026-04-11 10:39:52] INFO tekken.py:195: Non special vocabulary size is 130072 with 1000 special tokens.


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 17.79it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 22.44it/s]


In [4]:
openrouter_llm = OpenRouterLLM(
    model="openai/gpt-4o",
    api_key=openrouter_key,
)

In [7]:
await openrouter_llm.generate_response("Whats your name?")

{'response': 'I’m called ChatGPT. How can I assist you today?',
 'prompt_token_count': 11,
 'output_token_count': 13}

In [ ]:
api_key = json.loads(Path("../keys.json").read_text())["APILLM_KEY"]

api_llm_config = """backend: APILLM
name: gemini-3.1-flash-lite-preview
vertexai: true
generation_config:
  temperature: 1
  top_p: 0.95
  max_output_tokens: 65535
  safety_settings:
    - category: HARM_CATEGORY_HATE_SPEECH
      threshold: "OFF"
    - category: HARM_CATEGORY_DANGEROUS_CONTENT
      threshold: "OFF"
    - category: HARM_CATEGORY_SEXUALLY_EXPLICIT
      threshold: "OFF"
    - category: HARM_CATEGORY_HARASSMENT
      threshold: "OFF"
  thinking_config:
    thinking_level: LOW
"""

api_config_dict = yaml.safe_load(api_llm_config)
api_llm = create_llm_instance(api_config_dict, keys_file=api_key)

## Define Case Description

Set up the patient case that will be used across simulators.

In [4]:
# load the case description from data/extracted_profiles
name = "VS001"

case_description = json.loads(
    Path(f"../data/aci_bench/extracted_profiles/{name}.json").read_text()
)

transcript = parse_transcript_file(f"../data/aci_bench/conversations/{name}.txt")

## Initialize Patient Simulator

Choose which patient simulator to use and configure it.

In [5]:
baseline_patient = BaselinePatient(
    case_description=case_description,
    llm=api_llm,
    real_responses=[turn[1] for turn in transcript if turn[0] == "PATIENT"],
)

In [10]:
patient = PatientsWithPersonality(
    case_description=case_description,
    h=2,
    e=2,
    x=2,
    a=2,
    c=2,
    o=2,
    level="B",
    verbosity=2,
    llm=api_llm,
    meta_llm=meta_llm,
)

In [11]:
craftmd_patient = CraftMDPatient(
    case_description=case_description,
    llm=api_llm,
)

## Example Conversation

Try asking the patient some questions!

In [6]:
await baseline_patient.get_response("What brings you in today?")

'I am doing well, thanks for asking.'

In [12]:
await patient.simulate_conversation(
    conv=transcript, path=".", limit=1, align_doctor=True
)

[INIT] Performing lazy initialization...
[INIT] Generated downplayed leisure fields: {'tobacco': 'Unknown', 'alcohol': 'Unknown', 'illicit_drug': 'Unknown', 'sexual_history': "worried about the ol' wood not rising up", 'exercise': 'gets some steps in often'}
[INIT] Generated fuzzy medical history: {'allergies': ['Unknown'], 'family_medical_history': ['family members have had blood pressure issues'], 'medical_device': ['Unknown'], 'medical_history': ['blood pressure issues'], 'present_illness_positive': ["blood pressure issues that don't respond well to usual treatments", 'blood pressure is somewhat elevated', 'sometimes feels like heart is racing or acting up, had to go to ER once'], 'present_illness_negative': ['no unusual heart sensations lately', 'no discomfort or sensitivity detected', 'no noticeable swelling or fluid retention'], 'pain': ['no pain'], 'medication': ['blood pressure medication, not sure of exact type or dosage', 'blood pressure medication, not sure of the exact type

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[CLASSIFICATION] Model classified fields: []. Differing fields: set()
[CLASSIFICATION] Relevant fields: []

[SYSTEM PROMPT]
You are a 49-year-old male presenting with high blood pressure and palpitations. You are generally truthful but tend to downplay personal habits that might be embarrassing. You are health-conscious and concerned about your symptoms but are guarded in sharing information unless directly asked.

Your communication style is straightforward and concise, staying strictly on topic. You are naturally conversational but avoid unnecessary details or personal anecdotes. You may be skeptical of new treatments or technologies, preferring traditional or home remedies.

Your memory for dates and dosages is inconsistent—you may forget exact timelines or medication specifics unless prompted. You are suspicious of overly probing questions and may resist sharing information unless the doctor asks very specific, direct questions.

**Key tendencies under clinical questioning:**
- **D

,doctor_question,real_response,simulated_response
0,"Hello Mr. Roberts, how are you doing today?","I'm fine, thank you.","I’m doing alright, I suppose. Just dealing wit..."


In [13]:
patient.conversation_history

[{'role': 'user', 'content': 'Hello Mr. Roberts, how are you doing today?'},
 {'role': 'assistant',
  'content': "I’m doing alright, I suppose. Just dealing with these symptoms. My blood pressure's been up, and I've been getting some palpitations lately. That's why I'm here."}]